In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("YouTubeTopWatchSessions").getOrCreate()

data = [
    # video v1, user u1 session: 0 -> 120 = 120 sec
    ("e1", "2026-06-20T10:00:00Z", "u1", "v1", "play", 0),
    ("e2", "2026-06-20T10:05:00Z", "u1", "v1", "stop", 120),

    # video v1, user u2 session: 10 -> 400 = 390 sec
    ("e3", "2026-06-20T11:00:00Z", "u2", "v1", "play", 10),
    ("e4", "2026-06-20T11:10:00Z", "u2", "v1", "stop", 400),

    # video v1, user u3 session: 20 -> 250 = 230 sec
    ("e5", "2026-06-20T12:00:00Z", "u3", "v1", "play", 20),
    ("e6", "2026-06-20T12:07:00Z", "u3", "v1", "stop", 250),

    # video v1, user u4 session: 50 -> 600 = 550 sec
    ("e7", "2026-06-20T13:00:00Z", "u4", "v1", "play", 50),
    ("e8", "2026-06-20T13:20:00Z", "u4", "v1", "stop", 600),

    # video v2, user u1 session: 0 -> 80 = 80 sec
    ("e9", "2026-06-20T09:00:00Z", "u1", "v2", "play", 0),
    ("e10", "2026-06-20T09:03:00Z", "u1", "v2", "stop", 80),

    # video v2, user u2 session: 0 -> 300 = 300 sec
    ("e11", "2026-06-20T14:00:00Z", "u2", "v2", "play", 0),
    ("e12", "2026-06-20T14:08:00Z", "u2", "v2", "stop", 300),

    # video v2, user u3 session: 100 -> 350 = 250 sec
    ("e13", "2026-06-20T15:00:00Z", "u3", "v2", "play", 100),
    ("e14", "2026-06-20T15:06:00Z", "u3", "v2", "stop", 350),

    # Other date, should be ignored
    ("e15", "2026-06-21T10:00:00Z", "u1", "v1", "play", 0),
    ("e16", "2026-06-21T10:10:00Z", "u1", "v1", "stop", 500),
]

columns = [
    "event_id",
    "event_timestamp",
    "user_id",
    "video_id",
    "event_type",
    "playback_position_seconds"
]

df = spark.createDataFrame(data, columns)

df = df.withColumn(
    "event_timestamp",
    F.to_timestamp("event_timestamp")
)

df.createOrReplaceTempView("playback_events")

spark.sql("SELECT * FROM playback_events ORDER BY video_id, user_id, event_timestamp").show(truncate=False)

26/06/20 15:41:06 WARN Utils: Your hostname, Prems-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.43.0.101 instead (on interface en0)
26/06/20 15:41:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 15:41:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------+-------------------+-------+--------+----------+-------------------------+
|event_id|event_timestamp    |user_id|video_id|event_type|playback_position_seconds|
+--------+-------------------+-------+--------+----------+-------------------------+
|e1      |2026-06-20 17:00:00|u1     |v1      |play      |0                        |
|e2      |2026-06-20 17:05:00|u1     |v1      |stop      |120                      |
|e15     |2026-06-21 17:00:00|u1     |v1      |play      |0                        |
|e16     |2026-06-21 17:10:00|u1     |v1      |stop      |500                      |
|e3      |2026-06-20 18:00:00|u2     |v1      |play      |10                       |
|e4      |2026-06-20 18:10:00|u2     |v1      |stop      |400                      |
|e5      |2026-06-20 19:00:00|u3     |v1      |play      |20                       |
|e6      |2026-06-20 19:07:00|u3     |v1      |stop      |250                      |
|e7      |2026-06-20 20:00:00|u4     |v1      |play      |50     

In [2]:
# Your product manager wants a daily dashboard that shows the "Top 3 longest continuous watch sessions for each video, per day."
# A "continuous watch session" is defined as the difference in playback_position_seconds between a play event and the immediate next stop event for a given user on a given video. (For simplicity, assume every play is perfectly followed by a stop for now).
# 
# We have ingested this raw data into a large analytical data warehouse table named playback_events with the columns:
# event_id, event_timestamp, user_id, video_id, event_type, playback_position_seconds.
# 
# Your Task:
# Write a SQL query to find the top 3 longest watch sessions for each video_id for the date 2026-06-20.

In [4]:
def  run_sql(df):
    spark.sql(df).show()

In [29]:
sql1="""

with playback_events_buy_sell as (
SELECT 
    event_id,
    event_timestamp,
    user_id,
    video_id,
    event_type,
    playback_position_seconds,
    date(event_timestamp) event_date,
    Lead(playback_position_seconds) over (partition by  user_id,video_id order by event_timestamp asc, event_id asc ) AS next_playback_position_seconds,
    Lead(event_type) over (partition by  user_id,video_id order by event_timestamp asc, event_id asc) AS next_event_type
FROM playback_events
where event_type IN ('play', 'stop')
)
select 
event_id,
event_date,
video_id,
user_id,
event_type,
next_event_type,
playback_position_seconds,
next_playback_position_seconds,
next_playback_position_seconds-playback_position_seconds as diff_event_timestamp
FROM  playback_events_buy_sell
Where event_type='play' and next_event_type='stop'
AND next_playback_position_seconds-playback_position_seconds>120
AND  next_playback_position_seconds >= playback_position_seconds

"""
run_sql(sql1)

+--------+----------+--------+-------+----------+---------------+-------------------------+------------------------------+--------------------+
|event_id|event_date|video_id|user_id|event_type|next_event_type|playback_position_seconds|next_playback_position_seconds|diff_event_timestamp|
+--------+----------+--------+-------+----------+---------------+-------------------------+------------------------------+--------------------+
|     e15|2026-06-21|      v1|     u1|      play|           stop|                        0|                           500|                 500|
|      e3|2026-06-20|      v1|     u2|      play|           stop|                       10|                           400|                 390|
|     e11|2026-06-20|      v2|     u2|      play|           stop|                        0|                           300|                 300|
|      e5|2026-06-20|      v1|     u3|      play|           stop|                       20|                           250|              